# 🍐 Little Fig Studio

**Train any LLM on any hardware.** Select your model, configure, launch.

| Feature | Result |
|---|---|
| Quantization | Beats NF4 on 156/156 layers |
| GPU Speed | 7× faster than BnB NF4 |
| CPU Training | 1.1B model in 400MB RAM |
| Optimizer | FigMeZO (−18.6%, original research) |

**Run all cells below ↓**

In [ ]:
# Launch Little Fig Studio
import subprocess, time, sys
sys.path.insert(0, '/content/littlefig/src')

from pyngrok import ngrok

# Start server
server = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'little_fig.web.server:app',
     '--host', '0.0.0.0', '--port', '8888'],
    cwd='/content/littlefig/src',
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(4)

# Tunnel
public_url = ngrok.connect(8888)
print(f'\n🍐 Little Fig Studio is LIVE!')
print(f'\n   👉 {public_url}')
print(f'\n   Pick any model. Configure training. Launch.')
print(f'   Keep this cell running.')


In [ ]:
# Step 2: Launch Little Fig Studio
import subprocess, time, threading, re

# Start the web server
server = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'little_fig.web.server:app',
     '--host', '0.0.0.0', '--port', '8888'],
    cwd='/content/littlefig/src',
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(3)

# Check if server is running
import urllib.request
try:
    urllib.request.urlopen('http://localhost:8888', timeout=2)
    print('✅ Server running on port 8888')
except:
    # Server might return non-200 but still be alive
    print('⚠️ Server may not have started correctly. Check errors below:')
    try:
        err = server.stderr.read(1000).decode()
        print(err[:500])
    except:
        pass

# Start cloudflared tunnel (free, no account needed)
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8888'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(5)

# Read tunnel URL from stderr
tunnel_url = None
try:
    output = tunnel.stderr.read(4096).decode()
    urls = re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', output)
    if urls:
        tunnel_url = urls[0]
except:
    pass

if tunnel_url:
    print(f'\n🍐 Little Fig Studio is LIVE!')
    print(f'\n   👉 {tunnel_url}')
    print(f'\n   Open the link above in your browser.')
    print(f'   Select model → Configure → Train')
else:
    print('\n⚠️ Tunnel failed. Using iframe fallback...')
    from IPython.display import IFrame, display
    # Try Colab proxy
    from google.colab.output import eval_js
    proxy_url = eval_js("google.colab.kernel.proxyPort(8888)")
    print(f'   Try: {proxy_url}')
    display(IFrame(src=str(proxy_url), width='100%', height=700))

print('\n   Keep this cell running to keep the server alive.')

---
## Alternative: Python API (no UI)

Change `MODEL` to any HuggingFace model.

In [ ]:
from little_fig.engine import FigModel, FigTrainer, FigTrainingConfig

# === YOUR MODEL HERE ===
MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
# MODEL = 'google/gemma-3-4b-it'
# MODEL = 'Qwen/Qwen2.5-1.5B'
# MODEL = 'microsoft/phi-2'

model = FigModel.from_pretrained(MODEL, lora_r=16, lora_alpha=32, shared_codebook=True)
print(f'✅ {MODEL} loaded | Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

In [ ]:
# Train
config = FigTrainingConfig(
    num_epochs=1,
    learning_rate=2e-4,
    max_seq_length=256,
    batch_size=2,
    gradient_accumulation_steps=4,
    logging_steps=5,
    use_packing=True,
)
trainer = FigTrainer(model, config)
trainer.load_dataset('tatsu-lab/alpaca', max_samples=200)
trainer.train()
model.save_adapter('./my_adapter')
print('\n✅ Training complete! Adapter saved.')

---
*0xticketguy / Harboria Labs | AGPL-3.0*